In [2]:
### LLM 모델 기본 사용법과 프롬프트 전달방법 01
### Ollama 로컬 모델 사용

from langchain_ollama import ChatOllama

llm = ChatOllama(model="gemma4:e4b")

# llm.invoke("안녕하세요")
# llm.invoke("대한민국의 수도는?") # -> 이게 바로 프롬프트, 모델에게 전달하는 명령어 이자 질의 그자체


In [4]:
### 프롬프트 템플릿 사용

from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate(
    template="""
    {country}의 수도는?
    """,
    input_variables=["country"] ,
)

prompt_template.invoke({"country": "대한민국"}) # StringPromptValue(text='\n    대한민국의 수도는?\n    ')

llm.invoke(prompt_template.invoke({"country": "대한민국"}))



AIMessage(content='대한민국의 수도는 **서울**입니다.', additional_kwargs={}, response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-07-19T23:27:12.659614Z', 'done': True, 'done_reason': 'stop', 'total_duration': 11945717000, 'load_duration': 7035650125, 'prompt_eval_count': 22, 'prompt_eval_duration': 164114000, 'eval_count': 166, 'eval_duration': 4716607000, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'}, id='lc_run--019f7cb4-5467-74d2-ba2a-337a72b035a5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 22, 'output_tokens': 166, 'total_tokens': 188})

In [9]:
### 따라서 구조체 형식으로 뽑고 싶다면
### JsonOutputParser 가 아닌 llm.with_structured_output과 Pydantic Class를 같이 사용하면됨.

from typing import cast
from pydantic import BaseModel, Field

class CountryDetail(BaseModel):
    capital: str = Field(description="한 국가의 수도")

structured_llm = llm.with_structured_output(CountryDetail)

prompt_template = PromptTemplate(
    template="""
    {country}의 수도는? 수도 이름만 답해줘
    JSON 형식으로 답해줘
    """,
    input_variables=["country"]
)

llm.invoke(prompt_template.invoke({"country":"배트남"}))
resData:CountryDetail = cast(CountryDetail, structured_llm.invoke(prompt_template.invoke({"country":"배트남"})))
print(resData.capital)
print(resData.model_dump)




하노이
<bound method BaseModel.model_dump of CountryDetail(capital='하노이')>
